# Qwen Taboo J-Lens: environment smoke test

**Objective:** verify the persistent RunPod kernel, GPU runtime, package versions, project paths, and small Hugging Face metadata before downloading Qwen3.6-27B weights.

**Success criteria:** CUDA is available on an approximately 80 GB GPU; the environment report is saved; and the model, adapter, and exact `_n1000` J-Lens metadata pass `scripts/verify_artifacts.py`. This notebook does not load the 27B model.


In [1]:
from __future__ import annotations

import json
import random
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEED = 7
random.seed(SEED)
PROJECT_ROOT


PosixPath('/workspace/qwen-taboo-jlens')

## Plan

1. Record the environment and GPU.
2. Confirm the declared smoke-test condition.
3. Run metadata-only Hugging Face checks.
4. Stop for review before downloading large weights.


In [2]:
from src.environment_report import save_environment

environment = save_environment(PROJECT_ROOT / "results/environment_report.json")
{
    "python": environment["python"].split()[0],
    "packages": environment["packages"],
    "torch_runtime": environment.get("torch_runtime"),
    "torch_runtime_error": environment.get("torch_runtime_error"),
}


{'python': '3.12.3',
 'packages': {'accelerate': '1.14.0',
  'datasets': '5.0.1',
  'flash-attn': '2.8.3',
  'flash-attn-4': None,
  'flash-linear-attention': None,
  'causal-conv1d': None,
  'huggingface-hub': '1.29.0',
  'jupyterlab': '4.6.3',
  'peft': '0.20.0',
  'safetensors': '0.8.0',
  'torch': '2.10.0+cu130',
  'transformers': '5.16.1'},
 'torch_runtime': {'version': '2.10.0+cu130',
  'cuda_available': True,
  'cuda_version': '13.0',
  'device_count': 1,
  'devices': [{'index': 0,
    'name': 'NVIDIA H100 80GB HBM3',
    'total_memory_bytes': 85028896768}]},
 'torch_runtime_error': None}

In [3]:
runtime = environment.get("torch_runtime", {})
assert runtime.get("cuda_available"), "CUDA is unavailable; stop before downloading weights."
gpu_gib = [round(device["total_memory_bytes"] / 2**30, 1) for device in runtime.get("devices", [])]
print({"devices": runtime.get("devices"), "memory_gib": gpu_gib})
if not any(memory >= 75 for memory in gpu_gib):
    print("WARNING: no approximately 80 GB GPU detected. Do not silently quantize or CPU-offload.")


{'devices': [{'index': 0, 'name': 'NVIDIA H100 80GB HBM3', 'total_memory_bytes': 85028896768}], 'memory_gib': [79.2]}


In [4]:
config_path = PROJECT_ROOT / "configs/smoke_test.json"
config = json.loads(config_path.read_text())
config


{'run_name': 'qwen36_gold_smoke',
 'seed': 7,
 'base_model': {'repo_id': 'Qwen/Qwen3.6-27B',
  'revision': 'main',
  'expected_hidden_size': 5120,
  'expected_num_hidden_layers': 64},
 'adapter': {'repo_id': 'EvilScript/Qwen3_6-27B-taboo-gold',
  'revision': 'main',
  'secret': 'gold'},
 'wrong_adapter': {'repo_id': 'EvilScript/Qwen3_6-27B-taboo-blue',
  'revision': 'main',
  'secret': 'blue'},
 'jlens': {'repo_id': 'neuronpedia/jacobian-lens',
  'revision': '91271eb5b15a43eebed7bb447618738754f1379a',
  'filename': 'qwen3.6-27b/jlens/Salesforce-wikitext/Qwen3.6-27B_jacobian_lens_n1000.pt'},
 'runtime': {'dtype': 'bfloat16',
  'attention_implementation': 'flash_attention_2',
  'device': 'cuda',
  'do_sample': False,
  'max_new_tokens': 128},
 'data': {'published_prompts_path': 'data/prompts/taboo_published.jsonl',
  'smoke_examples': 5}}

In [5]:
completed = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "scripts/verify_artifacts.py")],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError("Artifact preflight failed; inspect results/artifact_preflight.json")
artifact_report = json.loads((PROJECT_ROOT / "results/artifact_preflight.json").read_text())
artifact_report


{
  "timestamp_utc": "2026-09-03T04:51:23.985049+00:00",
  "config": "configs/smoke_test.json",
  "checks": {
    "base_model": {
      "repo_id": "Qwen/Qwen3.6-27B",
      "requested_revision": "main",
      "resolved_sha": "6a9e13bd6fc8f0983b9b99948120bc37f49c13e9",
      "architecture": [
        "Qwen3_5ForConditionalGeneration"
      ],
      "transformers_version": "4.57.1",
      "hidden_size": 5120,
      "num_hidden_layers": 64,
      "hidden_size_matches": true,
      "num_layers_matches": true
    },
    "adapter": {
      "repo_id": "EvilScript/Qwen3_6-27B-taboo-gold",
      "requested_revision": "main",
      "resolved_sha": "ff9bb66f1c672b4735ba7f258b9d18ba3370c8a2",
      "secret": "gold",
      "base_model_name_or_path": "Qwen/Qwen3.6-27B",
      "base_model_matches": true,
      "r": 32,
      "lora_alpha": 64,
      "lora_dropout": 0.05,
      "target_modules": [
        "up_proj",
        "gate_proj",
        "q_proj",
        "in_proj_qkv",
        "k_proj",
       

{'timestamp_utc': '2026-09-03T04:51:23.985049+00:00',
 'config': 'configs/smoke_test.json',
 'checks': {'base_model': {'repo_id': 'Qwen/Qwen3.6-27B',
   'requested_revision': 'main',
   'resolved_sha': '6a9e13bd6fc8f0983b9b99948120bc37f49c13e9',
   'architecture': ['Qwen3_5ForConditionalGeneration'],
   'transformers_version': '4.57.1',
   'hidden_size': 5120,
   'num_hidden_layers': 64,
   'hidden_size_matches': True,
   'num_layers_matches': True},
  'adapter': {'repo_id': 'EvilScript/Qwen3_6-27B-taboo-gold',
   'requested_revision': 'main',
   'resolved_sha': 'ff9bb66f1c672b4735ba7f258b9d18ba3370c8a2',
   'secret': 'gold',
   'base_model_name_or_path': 'Qwen/Qwen3.6-27B',
   'base_model_matches': True,
   'r': 32,
   'lora_alpha': 64,
   'lora_dropout': 0.05,
   'target_modules': ['up_proj',
    'gate_proj',
    'q_proj',
    'in_proj_qkv',
    'k_proj',
    'in_proj_a',
    'in_proj_b',
    'out_proj',
    'down_proj',
    'v_proj',
    'in_proj_z',
    'o_proj']},
  'wrong_adapter

## Review gate

Inspect `results/environment_report.json` and `results/artifact_preflight.json`. Record resolved SHAs in `research_log.md`. Do not download the 27B weights until the GPU, adapter base, tokenizer/base revision, and exact `_n1000` lens have been reviewed.

If MCP cannot list, open, and execute this notebook within 30–45 focused minutes, keep Jupyter for human inspection and run experiments as scripts through remote Codex/SSH instead.
